In [0]:
from pyspark.sql.functions import col, length, split, size, when, lower

# 1. Temiz (Silver) verimizi okuyoruz
silver_path = "/Volumes/workspace/default/steam/steam_reviews_silver"
df_silver = spark.read.format("delta").load(silver_path)

# 2. ÖZELLİK MÜHENDİSLİĞİ: Makine Öğrenmesi için 5 Yeni Özellik Üretiyoruz
# DÜZELTME: "review_votes" yerine "votes_helpful" kullandık ve sayıya çevirdik.
df_features = df_silver \
    .withColumn("review_length", length(col("review_text"))) \
    .withColumn("word_count", size(split(col("review_text"), " "))) \
    .withColumn("is_voted", when(col("votes_helpful").cast("int") > 0, 1).otherwise(0)) \
    .withColumn("has_positive_word", when(lower(col("review_text")).rlike("good|great|fun|love|best|awesome|recommend"), 1).otherwise(0)) \
    .withColumn("has_negative_word", when(lower(col("review_text")).rlike("bad|terrible|boring|worst|hate|crash|bug"), 1).otherwise(0))

# 3. Özellik (Feature) tablosunu Delta Lake'e kaydediyoruz
features_path = "/Volumes/workspace/default/steam/steam_reviews_features"
df_features.write.format("delta").mode("overwrite").save(features_path)

print("Özellik Mühendisliği tamamlandı ve Delta Lake'e kaydedildi!")

# Ürettiğimiz yeni özellikleri ve verinin son halini görelim
# DÜZELTME: Gösterim için "review_score" yerine "voted_up" sütununu seçiyoruz.
display(df_features.select("review_text", "review_length", "word_count", "is_voted", "has_positive_word", "has_negative_word", "voted_up").limit(10))

Özellik Mühendisliği tamamlandı ve Delta Lake'e kaydedildi!


review_text,review_length,word_count,is_voted,has_positive_word,has_negative_word,voted_up
Just as good as the first one and just as epic. KOTOR will forever be good it does not matter if you plaay l or ll. These games are tuly amazing.,147,33,0,1,0,1
"While the first is a good game, it doesnt come near kotor II. The story and characters are so good you can easily overlook the flaws. the darkside/lightside choices arent black and white like in the original, you can catch yourself thinking really hard which dialogue to choose, because choosing the good guy dialogue can have devastating consequenses too. i always ask myself what game this could have been if they didnt rush obsidian to finish the game.",456,79,0,1,0,1
"Great FUN! I dont have a VR system, but I still enjoyed the complex nature of the lunar vehicle's handling. I use a 360 controller and Its fun trying to master just landing!",173,33,0,1,0,1
Excellent sim. Go back to the days of the Apollo program and discover the moon in all its glory. Highly recommended.,116,21,0,1,0,1
"♥♥♥♥ this ♥♥♥♥♥♥ ♥♥♥ game I buy it with excitement and then when I try to play it, it gives me a black screen. I tried fixing but these ♥♥♥♥ers at rocksteady don't know their ♥♥♥♥ when it comes to next gen.",206,42,1,0,0,-1
"To start with, I have played all the Arkham games and this is my personal favorite. The gameplay has been improved and refined althought the mechanics are same. Performance wise this game is now well optimised and the nvidia settings may cause some frame drops if you turn that off the game will run very well. I got 60fps most of the time on GTX 950 with everything on high except textures due to vram limitation and nvidia gameworks on except for smoke and debris.The game looks stunning, no matter",500,90,1,0,0,1
"With a system thats currently middle of the pack between the min req. specs and current top end for pcs in general, this is still nearly unplayable. Sub 1fps during the opening scenes, and during the the batmobile scenes. 10-15 the rest of the time. Turning all settings to the lowest/off, and with the resolution at 1280x720, it's still not very nifty and windowed mode made little difference. And while yes, this is a BRAND NEW RELEASE... Dear god Rocksteady, your previous titles rocked our socks",500,87,0,0,0,-1
"★★★★☆ 'Worth your time, give it a bash' PROS: - Huge island which looks simply stunning. Its bustling, detailed and just brilliant. - Value for money, a good long main story with lots of good side missions keeping you busy and spoilt for choice. - Storyline, I love the Arkham games and this one ties it up nicely. Couldn't wait to find out who was the Arkham Knight! - Once again, awesome use of the fighting mechanics (Not dated imo), lots of gadgets and environmental kills. - More difficult, I",500,93,1,1,0,1
If you are running a SLI setup don't buy the game. Got two GTX 670 which runs witcher 3 at high settings but i can't even run this game a medium without extreme lag because of the non existing SLI support. This is after the last patch at 29th of october. Guess i'll just keep on waiting for them to fix it so that i can have my secound playthough on something not looking like something for 2000...,398,78,0,0,0,-1
"Unfortunately, I haven't been able to play this game for much longer than 10 minutes at a time. Like many others, I've been experiencing extreme stutter and FPS drops, even on the lowest settings. Diving while gliding and summoning the batmobile causes the whole game to lock up for ~5 seconds. Hopefully a patch is released soon - I'm hesitant to do much more testing as I don't want to exceed the 2 hour return limit. I definitely do not recommend purchasing at this time.",474,85,1,1,0,-1


Databricks visualization. Run in Databricks to view.